[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/03-index-configuration/03-schema_json_serialization_and_reuse.ipynb)

# Schema JSON Serialization and Reuse

A `TableConfig` you build in a notebook lives only in that notebook's memory. The moment you restart the kernel, deploy to a different service, or hand the schema to a teammate, you'd have to rebuild it from scratch, field by field, character mapping by character mapping.

`TableConfig` solves this with `to_json()` and `from_json()`: the entire schema, including every attached `CharacterMapping` and `AliasSet`, serializes to a single file that can be version-controlled, shared, and loaded anywhere.

In this notebook you will:

1. Build a schema with harmonization attached, exactly as in the previous notebooks
2. Save it to a JSON file and inspect what actually gets written
3. Load it back in a simulated "different service"
4. Confirm the loaded schema behaves identically to the original
5. See the dictionary-based alternative for cases where a file on disk isn't the right fit


In [1]:
# !pip install mbox

## 1. Build a schema worth saving

Let's recreate the product schema from the first notebook in this directory, complete with the OCR-style character mapping and legacy code aliases attached to `product_id`. This is exactly the kind of schema you'd want to save rather than retype every time.

In [2]:
from mbox.config import TableConfig, TableFieldConfig, IndexType
from mbox.mapping import CharacterMapping
from mbox.aliases import AliasSet

product_id_mapping = CharacterMapping(
    name="product_id_ocr_cleaner",
    mapped_characters="A-Z0-9-",
    map_upper=True,
    deaccentuate=True,
    expand_umlauts=False,
    numbers_as_characters=True
)

legacy_code_aliases = AliasSet(name="legacy_product_codes")
legacy_code_aliases.add(word="B88-EXT", alias="LGCY-441", penalty=0)
legacy_code_aliases.add(word="A12-PWR", alias="LGCY-198", penalty=0)

schema = TableConfig(fields=[
    TableFieldConfig(
        column="product_id",
        index_type=IndexType.IDENT,
        character_mapping=product_id_mapping,
        aliases=legacy_code_aliases
    ),
    TableFieldConfig(column="product_name", index_type=IndexType.PHRASE),
    TableFieldConfig(column="unit_price", index_type=IndexType.DOUBLE)
])

schema.validate_fields()

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


True

## 2. Save the schema to JSON

`to_json()` writes the complete schema, including every nested `CharacterMapping` and `AliasSet`, to a single file.

In [3]:
import os

os.makedirs("schema", exist_ok=True)

schema.to_json("./schema/product_schema.json")
print("Schema saved.")

Schema saved.


Let's look at what actually got written. It's worth seeing this at least once, so you know a schema file isn't a black box, it's a plain, readable structure you could review in a pull request.

In [4]:
with open("./schema/product_schema.json") as f:
    print(f.read())

{"fields": [{"column": "product_id", "index_type": "Generic - Identity", "character_mapping": {"name": "product_id_ocr_cleaner", "mapped_characters": "A-Z0-9-", "map_upper": true, "deaccentuate": true, "expand_umlauts": false, "map_currency": true, "map_spaces": true, "any_to_latin": false, "numbers_as_characters": true}, "aliases": {"name": "legacy_product_codes", "aliases": {"{\"word\":\"B88-EXT\",\"alias\":\"LGCY-441\"}": 0, "{\"word\":\"A12-PWR\",\"alias\":\"LGCY-198\"}": 0}}}, {"column": "product_name", "index_type": "Generic - Phrase", "character_mapping": null, "aliases": null}, {"column": "unit_price", "index_type": "Numeric - Double", "character_mapping": null, "aliases": null}]}


Notice the `character_mapping` and `aliases` sections for `product_id` are fully nested inside the field's own entry, not stored separately and cross-referenced. The entire configuration for that column, type, spelling rules, and known synonyms together, lives in one self-contained block. This is the same completeness `TableFieldConfig` gave you in memory, just written to disk.

## 3. Load the schema in a "different service"

To make the point clearly, let's delete the schema object entirely, so nothing but the JSON file on disk remains, then load it back as if this were a completely separate process.

In [5]:
del schema
del product_id_mapping
del legacy_code_aliases

loaded_schema = TableConfig.from_json("./schema/product_schema.json")
loaded_schema

TableConfig(fields=[TableFieldConfig(column='product_id', index_type=<IndexType.IDENT: 'Generic - Identity'>, character_mapping=CharacterMapping(name='product_id_ocr_cleaner', mapped_characters='A-Z0-9-', map_upper=True, deaccentuate=True, expand_umlauts=False, map_currency=True, map_spaces=True, any_to_latin=False, numbers_as_characters=True), aliases=AliasSet(name='legacy_product_codes', aliases=OrderedDict({'{"word":"B88-EXT","alias":"LGCY-441"}': 0, '{"word":"A12-PWR","alias":"LGCY-198"}': 0}))), TableFieldConfig(column='product_name', index_type=<IndexType.PHRASE: 'Generic - Phrase'>, character_mapping=None, aliases=None), TableFieldConfig(column='unit_price', index_type=<IndexType.DOUBLE: 'Numeric - Double'>, character_mapping=None, aliases=None)])

No `TableFieldConfig`, no `CharacterMapping`, no `AliasSet` had to be reconstructed by hand. `from_json()` rebuilt the entire object graph from the file alone.

## 4. Confirm it behaves identically

Let's inspect the loaded field directly, and then build a real index, from `datasets/product_catalog_priced.csv`, to confirm the harmonization survived the round trip intact.

In [6]:
loaded_product_id_field = loaded_schema.get_field("product_id")
print("Index type:", loaded_product_id_field.index_type)
print("Has character mapping:", loaded_product_id_field.character_mapping is not None)
print("Has aliases:", loaded_product_id_field.aliases is not None)

Index type: IndexType.IDENT
Has character mapping: True
Has aliases: True


In [7]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.read_csv("datasets/product_catalog_priced.csv")

index = TableIndexer.create_index(
    df=df,
    config_overrides=loaded_schema,
    tmp_dir="tmp_index"
)

# The OCR mapping and legacy alias should both still work
ocr_query = index.match(product_id="8B8-3XT", include_field_scores=True)
legacy_query = index.match(product_id="LGCY-441", include_field_scores=True)

print("OCR-garbled query:")
display(ocr_query)

print("\nLegacy code query:")
display(legacy_query)

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.
OCR-garbled query:


,query_row,index_row,product_id_candidate,product_name_candidate,unit_price_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,24.99,100,100



Legacy code query:


,query_row,index_row,product_id_candidate,product_name_candidate,unit_price_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,24.99,100,100


Both queries should resolve exactly as they did when the schema was built directly in memory, in the earlier notebooks. Nothing about the harmonization behavior was lost by writing it to disk and loading it back.

## 5. When you need a dict instead of a file

Sometimes a schema needs to travel somewhere that isn't a filesystem path, an API response body, a message queue payload, a row in a database. `to_dict()` and `from_dict()` cover this without touching disk at all.

In [8]:
schema_dict = loaded_schema.to_dict()
print(type(schema_dict))

# Reconstruct from the dictionary, e.g. after receiving it from an API call
rebuilt_schema = TableConfig.from_dict(schema_dict)
rebuilt_schema.get_field("product_id").index_type

<class 'dict'>


<IndexType.IDENT: 'Generic - Identity'>

## 6. Recovering the schema from an auto-inferred index

Every schema so far was built explicitly, by hand, before the index was ever compiled. But what about an index built the simple way, with `index_columns` and no `config_overrides` at all, letting `TableIndexer` infer everything? Does that index have a schema at all, in the sense we've been working with?

It does. Every `TableIndex`, however it was built, keeps track of the exact configuration that was actually used to compile it, inferred or explicit. `get_index_config()` retrieves it:

```python
def get_index_config(self) -> TableConfig:
    return self._config
```

This means you never lose access to a schema just because you let inference handle it. Whatever `TableIndexer` decided, which `IndexType` each column got, is available afterward as a real `TableConfig` object, not just something that happened invisibly at compile time.

`to_json()` is really just `to_dict()` plus writing the result to a file, so the two are interchangeable depending on where the schema needs to end up next.

In [9]:
from mbox.indexing import TableIndexer
# Build an index the simple way, letting TableIndexer infer everything
inferred_index = TableIndexer.create_index(
    df=df,
    index_columns=["product_id", "product_name"],
    tmp_dir="tmp_index"
)

# Recover the schema TableIndexer actually decided on
inferred_schema = inferred_index.get_index_config()
inferred_schema.model_dump()

{'fields': [{'column': 'product_id',
   'index_type': <IndexType.IDENT: 'Generic - Identity'>,
   'character_mapping': {'name': 'Default Mapping',
    'mapped_characters': 'A-Za-z 0-9',
    'map_upper': True,
    'deaccentuate': True,
    'expand_umlauts': False,
    'map_currency': True,
    'map_spaces': True,
    'any_to_latin': False,
    'numbers_as_characters': False},
   'aliases': None},
  {'column': 'product_name',
   'index_type': <IndexType.PHRASE: 'Generic - Phrase'>,
   'character_mapping': {'name': 'Default Mapping',
    'mapped_characters': 'A-Za-z 0-9',
    'map_upper': True,
    'deaccentuate': True,
    'expand_umlauts': False,
    'map_currency': True,
    'map_spaces': True,
    'any_to_latin': False,
    'numbers_as_characters': False},
   'aliases': None},
  {'column': 'unit_price',
   'index_type': <IndexType.NON_SEARCHABLE: 'None - Not searchable, display only'>,
   'character_mapping': {'name': 'Default Mapping',
    'mapped_characters': 'A-Za-z 0-9',
    'map_

This is a real `TableConfig`, the same type you've been building by hand throughout this notebook. That means everything you already know how to do with a schema applies here too: inspect it with `get_field()`, save it with `to_json()`, and load it back with `from_json()`.

In [10]:
# Save the inferred schema exactly like any other TableConfig
inferred_schema.to_json("./schema/inferred_product_schema.json")

# Reload it, now it's an explicit, version-controlled schema,
# even though it started out as an automatic inference
restored_schema = TableConfig.from_json("./schema/inferred_product_schema.json")
restored_schema.get_field("product_name").model_dump()

{'column': 'product_name',
 'index_type': <IndexType.PHRASE: 'Generic - Phrase'>,
 'character_mapping': {'name': 'Default Mapping',
  'mapped_characters': 'A-Za-z 0-9',
  'map_upper': True,
  'deaccentuate': True,
  'expand_umlauts': False,
  'map_currency': True,
  'map_spaces': True,
  'any_to_latin': False,
  'numbers_as_characters': False},
 'aliases': None}

This is a useful pattern in practice: start with automatic inference while you're exploring a new dataset, and once you're happy with what M|BOX inferred, call `get_index_config()` and save the result. From that point on, you have an explicit, reviewable schema, without ever having written a single `TableFieldConfig` by hand.

## 7. A few practical notes

- **Commit schema JSON files alongside your code.** A schema is part of your application's contract with its data, treat it the same way you'd treat a database migration or an API schema, reviewed in pull requests, not regenerated silently.
- **Version your schema files if they change meaningfully.** `product_schema_v1.json`, `product_schema_v2.json`, or a version field inside your own deployment metadata, so you can tell which schema a given compiled index was built from.
- **Keep the schema and the DataFrame in sync.** Loading a schema doesn't validate that your current data actually has the columns it expects, `TableIndexer.create_index()` will raise an error if they don't line up, but it's worth checking deliberately in a pipeline rather than relying on that error to catch it.
- **A saved schema is a snapshot, not a live reference.** If you update a `CharacterMapping` in your code but forget to re-save the schema, `from_json()` will load the old version. There's no automatic sync between code and file.

## Next steps

You now have the full index-configuration toolkit: explicit field types, harmonization bundled directly into a field's config, dynamic schema management, and portable serialization. From here:

- **`04-recall-tuning/`** - once your index and schema are set, control exactly how much each field contributes to a match, and which algorithm compares it

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*